In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [ ]:
# ### 1. データの読み込みと基本確認
# 指定パターンのCSVをすべて読み込んで結合
files = Path(".").glob("bb_rsi_valley_summary_*_3rd_bid_4th_ASK_SETS.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df.head()


,Ticker,Company_Name,Last_Date,Last_Price,Stage,Signal,Detect_Date,Detect_Price,Note,RSI,...,BB_Days,BB_Close_Ratio,BB_Touch_Ratio,Lower_Shadow_Pct,Volume_Ratio,Exit_Dist_Pct,Upper_Shadow_Pct,Filter_Group_Id,Filter_Target_Index,Filter_Relative_Pos
0,1433.T,BESTERRA CO LTD,2026-06-08,1005.0,1st_RSI,BID,2026-02-02,1154.0,RSI crossed below 30 (oversold),29.77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,262,-2
1,1433.T,BESTERRA CO LTD,2026-06-08,1005.0,2nd_BB,BID,2026-02-03,1177.0,close<=-2σ & low touch >=50%,NaN,...,2.0,1.0,1.0,NaN,NaN,NaN,NaN,1,262,-1
2,1433.T,BESTERRA CO LTD,2026-06-08,1005.0,3rd_VALLEY,BID,2026-04-13,1051.0,Valley+LowerShadow+VolumeSpike,NaN,...,NaN,NaN,NaN,2.44,1.45,NaN,NaN,1,262,0
3,1433.T,BESTERRA CO LTD,2026-06-08,1005.0,4th_EXIT,ASK,2026-05-20,1006.0,+3σ 95% reach and upper shadow,NaN,...,NaN,NaN,NaN,NaN,NaN,-4.92,2.24,1,262,1
4,143A.T,ISHIN CO LTD,2026-06-05,684.0,1st_RSI,BID,2025-12-26,785.0,RSI crossed below 30 (oversold),25.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,298,-2


In [ ]:
# ### 2. 特徴量の統合 (Feature Engineering)
features = []

# TickerとGroup_Idの両方でグループ化（複数ファイル結合時のID重複対策）
for (ticker, group_id), group in df.groupby(["Ticker", "Filter_Group_Id"]):
    # 必要なステージを抽出
    stages = {s: group[group["Stage"] == s] for s in ["1st_RSI", "2nd_BB", "3rd_VALLEY", "4th_EXIT"]}

    # Valley(買い)とExit(売り)の両方が揃っている場合のみ処理
    if not stages["3rd_VALLEY"].empty and not stages["4th_EXIT"].empty:
        buy_price = stages["3rd_VALLEY"]["Detect_Price"].values[0]
        sell_price = stages["4th_EXIT"]["Detect_Price"].values[0]
        profit_pct = ((sell_price - buy_price) / buy_price) * 100

        # 基本となる行（Valley）を作成
        feature_row = stages["3rd_VALLEY"].iloc[0].to_dict()

        # 他のステージにある欠損値を補完（RSIやBBの指標を統合）
        for s_name, s_df in stages.items():
            if not s_df.empty:
                for col, val in s_df.iloc[0].to_dict().items():
                    if pd.isna(feature_row.get(col)):
                        feature_row[col] = val

        feature_row["Group_Id"] = group_id
        feature_row["Buy_Price"] = buy_price
        feature_row["Profit_Pct"] = profit_pct
        feature_row["Target_Profit"] = 1 if profit_pct >= 5 else 0
        features.append(feature_row)

features_df = pd.DataFrame(features).dropna(subset=["Target_Profit"]).dropna(axis=1, how="all")

# 重複排除: 同銘柄かつDetect_Dateが同じものを削除
initial_count = len(features_df)
features_df = features_df.drop_duplicates(subset=['Ticker', 'Detect_Date'])
final_count = len(features_df)

# 学習・保存に不要な特徴量を削除
drop_cols = [
    "Exit_Dist_Pct",
    "Upper_Shadow_Pct",
    "Filter_Group_Id",
    "Filter_Target_Index",
    "Group_Id"
]
features_df = features_df.drop(columns=drop_cols, errors='ignore')

print(f"元のサンプル数: {initial_count}")
print(f"重複排除後のサンプル数: {final_count} (削除数: {initial_count - final_count})")
print(features_df.head())
features_df.to_csv("brv_datasets.csv", index=False)

元のサンプル数: 684
重複排除後のサンプル数: 166 (削除数: 518)
   Ticker     Company_Name   Last_Date  Last_Price       Stage Signal  \
0  1433.T  BESTERRA CO LTD  2026-06-08      1005.0  3rd_VALLEY    BID   
1  143A.T     ISHIN CO LTD  2026-06-05       684.0  3rd_VALLEY    BID   
2  143A.T     ISHIN CO LTD  2026-06-05       684.0  3rd_VALLEY    BID   
3  1444.T    NISSOU CO LTD  2026-06-05      2834.0  3rd_VALLEY    BID   
5  150A.T       JSH CO LTD  2026-06-08       337.0  3rd_VALLEY    BID   

  Detect_Date  Detect_Price                            Note    RSI  ...  \
0  2026-04-13        1051.0  Valley+LowerShadow+VolumeSpike  29.77  ...   
1  2026-02-25         818.0  Valley+LowerShadow+VolumeSpike  25.02  ...   
2  2026-03-05         763.0  Valley+LowerShadow+VolumeSpike  29.54  ...   
3  2026-03-18        2682.0  Valley+LowerShadow+VolumeSpike  20.42  ...   
5  2026-05-14         340.0  Valley+LowerShadow+VolumeSpike  27.97  ...   

   RSI_Zscore20  BB_Days  BB_Close_Ratio  BB_Touch_Ratio  Lower_Shado

In [ ]:
features_df.columns


Index(['Ticker', 'Company_Name', 'Last_Date', 'Last_Price', 'Stage', 'Signal',
       'Detect_Date', 'Detect_Price', 'Note', 'RSI', 'Prev_RSI', 'RSI_Diff',
       'RSI_Drop_Pct', 'RSI_30_Gap', 'RSI_20_Gap', 'RSI_Depth',
       'RSI_Down_Streak', 'RSI_Rank20', 'RSI_Zscore20', 'BB_Days',
       'BB_Close_Ratio', 'BB_Touch_Ratio', 'Lower_Shadow_Pct', 'Volume_Ratio',
       'Filter_Relative_Pos', 'Buy_Price', 'Profit_Pct', 'Target_Profit'],
      dtype='object')

In [ ]:
keep_cols = [
    "RSI_Diff",
    "RSI_20_Gap",
    "Lower_Shadow_Pct",
    "BB_Days",
    "Volume_Ratio",
]
X = features_df[keep_cols].fillna(0)
y = features_df["Target_Profit"]

In [ ]:
# 学習前に特徴量の確認
print("=== 学習に使用する特徴量 ===")
print(X.columns.tolist())

=== 学習に使用する特徴量 ===
['RSI_Diff', 'RSI_20_Gap', 'Lower_Shadow_Pct', 'BB_Days', 'Volume_Ratio']


In [ ]:
# 学習前の評価
print("=== 学習前のターゲット分布 ===")
print(y.value_counts())

=== 学習前のターゲット分布 ===
Target_Profit
0    109
1     57
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

tree_clf = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=10,
    random_state=42,
    class_weight="balanced"
)

tree_clf.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current n

In [ ]:
print(
    classification_report(
        y_test,
        tree_clf.predict(X_test)
    )
)

              precision    recall  f1-score   support

           0       0.69      0.27      0.39        33
           1       0.35      0.76      0.48        17

    accuracy                           0.44        50
   macro avg       0.52      0.52      0.44        50
weighted avg       0.58      0.44      0.42        50



In [ ]:
print(f"訓練精度: {accuracy_score(y_train, tree_clf.predict(X_train)):.3f}")
print(f"テスト精度: {accuracy_score(y_test, tree_clf.predict(X_test)):.3f}")

訓練精度: 0.534
テスト精度: 0.440


In [ ]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree_clf.feature_importances_
})

importance_df.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
4,Volume_Ratio,0.605495
1,RSI_20_Gap,0.394505
0,RSI_Diff,0.000000
2,Lower_Shadow_Pct,0.000000
3,BB_Days,0.000000
